# BGMI Performance Monitor - Build APK on Google Colab

This notebook compiles `main.py` (Kivy app) into an installable Android `.apk` using **Buildozer**, entirely in the cloud.

You do **NOT** need WSL, Linux, Android Studio, or anything installed on your own PC - everything below runs on Google's free Colab servers via Python.

### How to use
1. Open this file in https://colab.research.google.com (File > Upload notebook).
2. Run the cells **one by one**, from top to bottom.
3. When asked, upload `main.py` and `buildozer.spec` (the cell will prompt a file picker).
4. Wait ~15-25 minutes for the first build (downloads Android SDK/NDK).
5. The last cell downloads the finished `bgmimonitor-1.0-arm64-v8a_armeabi-v7a-debug.apk` to your computer.

In [ ]:
# STEP 1: Install Buildozer + Cython
!pip install buildozer
!pip install cython==0.29.33

In [ ]:
# STEP 2: Install required system packages for python-for-android
!sudo apt-get update
!sudo apt-get install -y python3-pip build-essential git python3 python3-dev \
    ffmpeg libsdl2-dev libsdl2-image-dev libsdl2-mixer-dev libsdl2-ttf-dev \
    libportmidi-dev libswscale-dev libavformat-dev libavcodec-dev zlib1g-dev \
    libgstreamer1.0-dev gstreamer1.0-plugins-base gstreamer1.0-plugins-good \
    libffi-dev libtool automake autoconf openjdk-17-jdk unzip

In [ ]:
# STEP 3: Create a clean project folder and upload your files into it
import os
os.makedirs('/content/bgmi_app', exist_ok=True)
os.chdir('/content/bgmi_app')

from google.colab import files
print('Please choose BOTH: main.py AND buildozer.spec')
uploaded = files.upload()

In [ ]:
# STEP 4: Build the debug APK (this step takes a while the first time)
!buildozer -v android debug

In [ ]:
# STEP 5: Locate & download the generated APK
import glob
from google.colab import files as gfiles

apks = glob.glob('/content/bgmi_app/bin/*.apk')
print('Found APKs:', apks)
if apks:
    gfiles.download(apks[0])
else:
    print('No APK found yet - scroll up and check the build log for errors.')

### Troubleshooting
- If the build fails with a Cython error, try `!pip install cython==0.29.19` or `cython==3.0.6` instead, then re-run Step 4.
- If it fails on a missing package, `!sudo apt-get install <package>` then re-run Step 4.
- Run `!buildozer android clean` if you change `buildozer.spec` and need a fresh rebuild.
- The first build downloads the Android SDK/NDK (~1-2 GB) so it can take 15-25 minutes; later builds are much faster.